# TP3 – Retail Intelligence
## Étape 1 : Préparation de l'environnement

Objectif : Vérifier le fonctionnement de Python, Apache Spark et PySpark, puis réaliser un premier traitement sur un fichier CSV de test.

Mode d'exécution : Spark local sur Ubuntu.

In [1]:
import sys
import pyspark

from pyspark.sql import SparkSession

print("Version Python :", sys.version)
print("Version PySpark :", pyspark.__version__)

Version Python : 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
Version PySpark : 4.3.0


In [2]:
spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .getOrCreate()
)

print("Version Spark :", spark.version)
print("Session Spark :", spark.sparkContext.appName)
print(spark.sparkContext.master)
print(spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS"))
print("Session Spark créée avec succès !")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 17:09:28 WARN Utils: Your hostname, NAVID-DELL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/18 17:09:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 17:09:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/18 17:09:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/18 17:09:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Version Spark : 4.3.0
Session Spark : TP3-Retail-Intelligence
local[*]
file:///
Session Spark créée avec succès !


In [3]:
import os
import getpass

utilisateur = getpass.getuser()

chemin_fichier = f"file:///home/{utilisateur}/retail_project/data/customer/customer.dat"

print(chemin_fichier)

df = (
    spark.read
    .option("header", False)
    .option("inferSchema", True)
    .option("sep", "|")
    .csv(chemin_fichier)
)

df.show()
df.printSchema()
print("Nombre de lignes :", df.count())

file:///home/navid/retail_project/data/customer/customer.dat
+---+----------------+-------+----+-----+-----+------+----+--------+----------+----+----+----+----+--------------------+-------------+--------------------+----------+
|_c0|             _c1|    _c2| _c3|  _c4|  _c5|   _c6| _c7|     _c8|       _c9|_c10|_c11|_c12|_c13|                _c14|         _c15|                _c16|      _c17|
+---+----------------+-------+----+-----+-----+------+----+--------+----------+----+----+----+----+--------------------+-------------+--------------------+----------+
|  0|AAAAAAAAAAAAAAAA|1824793|3203| 2555|28776| 14690| Ms.|  Marisa|Harrington|   N|  17|   4|1988|UNITED ARAB EMIRATES| RRCyuY3XfE3a|Marisa.Harrington...|  gdMmGdU9|
|  1|AAAAAAAAAAAAAAAB| 830976|2600| 9191|94658| 19931|Miss|  Bessie|  Calderon|   N|  20|  10|1945|              MONACO|      aCJb9cc|Bessie.Calderon@s...| hLQn0LtVa|
|  2|AAAAAAAAAAAAAAAC| 335540|5527|14091|31783|106195|Miss| Barbara|    Hoover|   Y|   3|   4|1952|     

In [4]:
chemin_sortie = "/home/" + __import__("getpass").getuser() + "/retail-intelligence/tp3/output/test_customer_parquet"

df.write.mode("overwrite").parquet(chemin_sortie)

print("Écriture Parquet réalisée avec succès.")

[Stage 6:>                                                          (0 + 4) / 4]

Écriture Parquet réalisée avec succès.


In [5]:
df_parquet = spark.read.parquet(chemin_sortie)

df_parquet.show()
print("Nombre de lignes :", df_parquet.count())

+---+----------------+-------+----+-----+-----+------+----+--------+----------+----+----+----+----+--------------------+-------------+--------------------+----------+
|_c0|             _c1|    _c2| _c3|  _c4|  _c5|   _c6| _c7|     _c8|       _c9|_c10|_c11|_c12|_c13|                _c14|         _c15|                _c16|      _c17|
+---+----------------+-------+----+-----+-----+------+----+--------+----------+----+----+----+----+--------------------+-------------+--------------------+----------+
|  0|AAAAAAAAAAAAAAAA|1824793|3203| 2555|28776| 14690| Ms.|  Marisa|Harrington|   N|  17|   4|1988|UNITED ARAB EMIRATES| RRCyuY3XfE3a|Marisa.Harrington...|  gdMmGdU9|
|  1|AAAAAAAAAAAAAAAB| 830976|2600| 9191|94658| 19931|Miss|  Bessie|  Calderon|   N|  20|  10|1945|              MONACO|      aCJb9cc|Bessie.Calderon@s...| hLQn0LtVa|
|  2|AAAAAAAAAAAAAAAC| 335540|5527|14091|31783|106195|Miss| Barbara|    Hoover|   Y|   3|   4|1952|           GREENLAND|   59cfL9Fbpv|Barbara.Hoover@la...|  vDVbSOCI

In [6]:
spark.stop()
print("Session Spark fermée.")

Session Spark fermée.
